In [24]:
from google.colab import drive
drive.mount('/content/drive')

#load the data from my drive

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/first_25000_rows.csv')
df.head()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,ask_sz_08,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol
0,2024-10-21T11:54:29.221230963Z,2024-10-21T11:54:29.221064336Z,10,2,38,C,B,1,233.62,2,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
1,2024-10-21T11:54:29.223936626Z,2024-10-21T11:54:29.223769812Z,10,2,38,A,B,0,233.67,2,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
2,2024-10-21T11:54:29.225196809Z,2024-10-21T11:54:29.225030400Z,10,2,38,A,B,0,233.67,3,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
3,2024-10-21T11:54:29.712600612Z,2024-10-21T11:54:29.712434212Z,10,2,38,A,B,2,233.52,200,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
4,2024-10-21T11:54:29.764839221Z,2024-10-21T11:54:29.764673165Z,10,2,38,C,B,2,233.52,200,...,155,1,7,233.25,234.13,55,400,2,1,AAPL


Task 1: Order Flow Imbalances (OFI)

In [32]:
import pandas as pd

# ========================
# BEST LEVEL OFI
# ========================

# depth 0 is the best level I think
df_best = df[df['depth'] == 0]

# only actions that affect queue
df_best = df_best[df_best['action'].isin(['A', 'C'])]

# bid/ask side map: B = bid = 1, A = ask = -1
df_best['side_sign'] = df_best['side'].map({'B': 1, 'A': -1})

# add = 1, cancel = -1
df_best['action_sign'] = df_best['action'].map({'A': 1, 'C': -1})

# this should give the signed OFI value
df_best['ofi'] = df_best['size'] * df_best['side_sign'] * df_best['action_sign']


# sum per timestamp
best_ofi = df_best.groupby('ts_recv')['ofi'].sum().reset_index()
best_ofi.rename(columns={'ofi': 'best_level_ofi'}, inplace=True)



# preview
print("Best-Level OFI example:")
print(best_ofi.head(10))



Best-Level OFI example:
                          ts_recv  best_level_ofi
0  2024-10-21T11:54:29.223936626Z             2.0
1  2024-10-21T11:54:29.225196809Z             3.0
2  2024-10-21T11:54:37.990960617Z          -200.0
3  2024-10-21T11:54:39.124458000Z           200.0
4  2024-10-21T11:54:39.134307134Z          -200.0
5  2024-10-21T11:54:41.233347439Z            -3.0
6  2024-10-21T11:54:41.234786818Z             1.0
7  2024-10-21T11:54:41.236073416Z            -1.0
8  2024-10-21T11:54:41.236144516Z            -2.0
9  2024-10-21T11:54:41.238784694Z             2.0


In [33]:
# ========================
# MULTI LEVEL OFI (depth 0 to 5)
# ========================

# tried depth 10 but too messy, going with 0-5
df_multi = df[df['depth'] <= 5]
df_multi = df_multi[df_multi['action'].isin(['A', 'C'])]

df_multi['side_sign'] = df_multi['side'].map({'B': 1, 'A': -1})
df_multi['action_sign'] = df_multi['action'].map({'A': 1, 'C': -1})

# adding some depth-based weighting (shallow = more important)
df_multi['depth_weight'] = 1 / (df_multi['depth'] + 1)

# weighted OFI
df_multi['ofi'] = df_multi['size'] * df_multi['side_sign'] * df_multi['action_sign'] * df_multi['depth_weight']

# aggregate per time
multi_ofi = df_multi.groupby('ts_recv')['ofi'].sum().reset_index()
multi_ofi.rename(columns={'ofi': 'multi_level_ofi'}, inplace=True)

# just checking
print("Multi-Level OFI preview:")
print(multi_ofi.head(10))


Multi-Level OFI preview:
                          ts_recv  multi_level_ofi
0  2024-10-21T11:54:29.221230963Z        -1.000000
1  2024-10-21T11:54:29.223936626Z         2.000000
2  2024-10-21T11:54:29.225196809Z         3.000000
3  2024-10-21T11:54:29.712600612Z        66.666667
4  2024-10-21T11:54:29.764839221Z       -66.666667
5  2024-10-21T11:54:29.764851707Z       200.000000
6  2024-10-21T11:54:37.990960617Z      -200.000000
7  2024-10-21T11:54:39.124458000Z       200.000000
8  2024-10-21T11:54:39.134307134Z      -200.000000
9  2024-10-21T11:54:41.233347439Z        -3.000000


In [34]:
# ========================
# INTEGRATED OFI (rolling window over best level)
# ========================

# gotta sort timestamps first
best_ofi = best_ofi.sort_values('ts_recv')

# I’ll just use 10 rows as a window for now
best_ofi['integrated_ofi'] = best_ofi['best_level_ofi'].rolling(window=10).sum()

print("Integrated OFI example:")
print(best_ofi[['ts_recv', 'integrated_ofi']].head(15))


Integrated OFI example:
                           ts_recv  integrated_ofi
0   2024-10-21T11:54:29.223936626Z             NaN
1   2024-10-21T11:54:29.225196809Z             NaN
2   2024-10-21T11:54:37.990960617Z             NaN
3   2024-10-21T11:54:39.124458000Z             NaN
4   2024-10-21T11:54:39.134307134Z             NaN
5   2024-10-21T11:54:41.233347439Z             NaN
6   2024-10-21T11:54:41.234786818Z             NaN
7   2024-10-21T11:54:41.236073416Z             NaN
8   2024-10-21T11:54:41.236144516Z             NaN
9   2024-10-21T11:54:41.238784694Z          -198.0
10  2024-10-21T11:54:41.239993456Z          -197.0
11  2024-10-21T11:54:53.247698803Z          -203.0
12  2024-10-21T11:54:53.249060172Z            -5.0
13  2024-10-21T11:54:53.251488043Z          -202.0
14  2024-10-21T11:54:53.252712795Z             0.0


In [35]:
# ========================
# CROSS-ASSET OFI (only if we have multiple symbols)
# ========================

if 'symbol' in df.columns and df['symbol'].nunique() > 1:
    df_cross = df[df['depth'] == 0]
    df_cross = df_cross[df_cross['action'].isin(['A', 'C'])]

    df_cross['side_sign'] = df_cross['side'].map({'B': 1, 'A': -1})
    df_cross['action_sign'] = df_cross['action'].map({'A': 1, 'C': -1})
    df_cross['ofi'] = df_cross['size'] * df_cross['side_sign'] * df_cross['action_sign']

    ofi_by_sym = df_cross.groupby(['symbol', 'ts_recv'])['ofi'].sum().reset_index()

    all_cross = []

    for sym in ofi_by_sym['symbol'].unique():
        one = ofi_by_sym[ofi_by_sym['symbol'] == sym]
        others = ofi_by_sym[ofi_by_sym['symbol'] != sym]

        # sum others’ OFI by time
        cross_sum = others.groupby('ts_recv')['ofi'].sum().reset_index()
        cross_sum.rename(columns={'ofi': 'cross_asset_ofi'}, inplace=True)

        temp = pd.merge(one, cross_sum, on='ts_recv', how='left')
        temp['symbol'] = sym
        all_cross.append(temp)

    final_cross_ofi = pd.concat(all_cross)
    print("Cross-Asset OFI sample:")
    print(final_cross_ofi.head(10))
else:
    print("Only one symbol in data, skipping cross-asset OFI.")

Only one symbol in data, skipping cross-asset OFI.


In [36]:
# ========================
# FINAL JOINED RESULT (for clear output to see and read :) )
# ========================

# merge multi and integrated into best
final = pd.merge(best_ofi, multi_ofi, on='ts_recv', how='left')

# if cross-asset exists, add it too
if 'final_cross_ofi' in locals():
    final = pd.merge(final, final_cross_ofi[['ts_recv', 'cross_asset_ofi']], on='ts_recv', how='left')

print("Final combined result:")
print(final.head(10))

Final combined result:
                          ts_recv  best_level_ofi  integrated_ofi  \
0  2024-10-21T11:54:29.223936626Z             2.0             NaN   
1  2024-10-21T11:54:29.225196809Z             3.0             NaN   
2  2024-10-21T11:54:37.990960617Z          -200.0             NaN   
3  2024-10-21T11:54:39.124458000Z           200.0             NaN   
4  2024-10-21T11:54:39.134307134Z          -200.0             NaN   
5  2024-10-21T11:54:41.233347439Z            -3.0             NaN   
6  2024-10-21T11:54:41.234786818Z             1.0             NaN   
7  2024-10-21T11:54:41.236073416Z            -1.0             NaN   
8  2024-10-21T11:54:41.236144516Z            -2.0             NaN   
9  2024-10-21T11:54:41.238784694Z             2.0          -198.0   

   multi_level_ofi  
0              2.0  
1              3.0  
2           -200.0  
3            200.0  
4           -200.0  
5             -3.0  
6              1.0  
7             -1.0  
8             -2.0  
9      